Vision-Lanuage Model Turorial

This tutorial was created using the following as a guide: https://huggingface.co/blog/AviSoori1x/seemore-vision-language-model. However, the notes created on the text is my own writing.

In [ ]:
# imports
import torch.nn as nn
import torch
import torch.nn.functional as F

# 1. Define device dynamically based on hardware availability
if torch.cuda.is_available():
    device = torch.device("cuda")  # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device("mps")   # Apple Silicon GPU (M1/M2/M3/M4)
else:
    device = torch.device("cpu")   # Default Fallback


In [ ]:
# creating a patch embedding class. This is important for taking an image and creating a sequence of patches with it. THis will help the transformer process the visiual data.
class PatchEmbeddings(nn.Module):
    # nnModule relates to the neural net module in pytorch
    def __init__(self, img_size=96, patch_size=6, hidden_dim=512):
        # represents the image size and patch size in pixels. The hidden_dims represent how many dimensions the vector that represents a singular patch will be. This is important as the RGB values inside the patch will be multiplied by some number of weights to get this vector that represents what the model finds important in that patch of the image.
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size//patch_size)**2 # squared because each image is treated like a square

        # creating convolutional layer to extract patch embeddings (like a CNN)
        self.conv = nn.Connv2d(in_channels=3, out_channels=hidden_dim, kernal_size=patch_size, stride=patch_size)
        # in_channels: the RGB values that are stored in the images
        # out_channel: what the CNN needs to retrun that represents the important information from the image (size is hidden_dims due to explaination above)
        # kernal_size/patch_size: indicate how big the patch is and to make sure each patch is embedded separately (independent)

    def forward(self, X):
        # this is the function where the actual computation happens. The image will be split into patches and linearly projected (as setup in the init function)
        # gets the patch embeddings from the input image (using the CNN)
        X = self.conv(X)

        X = X.flatten(2)
        # focuses on reshaping the array to be [batch_size, num_patches, hidden_dim]
        X = X.transpose(1, 2) # this specific line works to change the position of num_patches to the 2nd position

        return X

In [ ]:
# creating the attention heads (implementing causal self attention and multi-head casual attention)
# important to note that this attention mechanism has been developed to be used for both the decorder portion and the vision transformer in this VLM implementation. This is important to note because the decoder transformer will use causal attention (meaning that it will be blocked from seeing tokens that are beyond the model's current position) yet the vision model will not be using this casual attention mechanism
# the blog states that the reason the decoder uses casual self-attention is because it helps ensure the interity of the autoregressive language generation. This because the basis of autoregressive language generation is that the input is only previous tokens, not future tokens. Also this is critical in the training of the prediction model because if the model could just look at the entire text, then there would be no need for a prediciton aspect -- it could just return the next token

class Head(nn.Module):
    def __init__(self, n_embd, head_size, dropout=0.1, is_decoder=False):
        # n_embd represents the size of the input and from every block in the transformer structure. This allows vector computation to be conducted.
        # head_size represents how much of the n_embd sized vector each head (from the multi-head attention mechanism) will get to analyze. In the end, the output from each of these attention mechanisms will be concatted to form the final output (so it says the same size as n_embd)
        # dropout is a regularization setting that works to zero out some random values in the model. This allows the model to prevent overfitting and makes the model more robust
        # is_decoder checks if the decoder is using the strcutre to activate the appropriate attention mechanism

        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.dropout = nn.Dropout(dropout)
        self.is_decoder = is_decoder

    def forward(self, x):
        # gets the batch size, sequence length, and embedding dimensions from the input 
        B, T, C = x.shape

        # calculates the key, query, and value projections (shape: B, T, head_size)
        # Query: vector that works to extract specific information from each token/embedding
        # Key: vector that stores what information each token actual stroes (like a label/tag) the query can be matched against
        # Value: based on the query-key matches, each of the words develop a new embedding, dependent on their relationships to other words
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # compute the attention scores (the formula: dot product of query and scaling by the square root of the embedding dimension)
        wei = q @ k.transpose(-2, -1) * (C ** -0.5) # (shape: B, T, T)

        if self.is_decoder:
            # need to apply the causal mask
            # tril represents a large T by T matrix with true values only in the lower triangular half of the matrix (exactly what the causual attention is)
            tril = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
            # now wei is adjusted such that only the values in the lower triangular are kept and the rest of the values are made as 0 (therefore they get 0 attention once the softmax is applied)
            wei = wei.masked_fill(tril == 0, float('-inf'))

        # creates a probability distribution to display how similar/the relationship between the words
        # it creates the proability distribuition per batch and per query. this means per row it is going to determine how much each key lines up with the query and these values are going to be put between 0 and 1.
        wei = F.softmax(wei, dim=-1) # shape: B, T, T

        wei = self.dropout(wei)

        # performs weighted aggredgation of values to restructure the embeddings of each word to represent their relationship to other words
        out = wei @ v 

        return out

In [ ]:
# mutlihead attention implementation

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, num_heads, dropout=0.1, is_decoder=False):
        super().__init__()

        assert n_embd % num_heads == 0, "n_embd must be divisible by num_heads"

        # create a list of attention heads (each head is going to be incharge of understanding a different relationship)
        self.heads = nn.ModuleList([
            Head(n_embd, n_embd//num_heads, dropout, is_decoder)
            for _ in range(num_heads)
        ])

        # linear layer for projecting the concatenated head outputs
        # works to merge the findings from each of the heads instead of having them just be numbers that were added together
        self.proj = nn.Linear(n_embd, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # applying attention on the input tensor
        head_outputs = [h(x) for h in self.heads]

        out = torch.cat(head_outputs, dim=-1)

        out = self.proj(out)

        out = self.dropout(out)

        return out     

: 

In [ ]:
# multilayer perception
class MLP(nn.Module):
    def __init__(self, n_embd, dropout=0.1, is_decoder=True):
        super().__init__()

        # layers of the MLP (linear and non-linear)
        layers = [
            # input layer that takes in the n_embd and outputs it to a layer that 4*n_embd
            nn.Linear(n_embd, 4* n_embd),

            nn.ReLU() if is_decoder else nn.GELU(), 
            # different activation functions for the vision vs decoder modules

            nn.Linear(4*n_embd, n_embd), 

            nn.Dropout(dropout)
        ]

        # sequential container to hold the layers
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # pass input through the MLP
        return self.net(x)

In [ ]:
# block module for the transformer
class Block(nn.Module):
    def __init__(self, n_embd, num_heads, dropout=0.1, is_decoder=False):
        super().__init__()

        # normalize the input going into the model
        self.ln1 = nn.LayerNorm(n_embd)

        # attention
        self.attn = MultiHeadAttention(n_embd, num_heads, dropout, is_decoder)

        # normalize input into next step
        self.ln3 = nn.LayerNorm(n_embd)

        # feed-forward nueral network/MLP (differs from the tutorial)
        self.ffn = MLP(n_embd, num_heads, dropout, is_decoder)

    def forward(self, x):
        # save input
        original_x = x

        # applying layer normalization
        x = self.ln1(x)

        # apply multi-head attention
        attn_output = self.attn(x)

        # add residual connection to attention output; it is a way to sort of retain history of what has already been discovered from the previous layers; this way the current layer does not need to recompute everything perfectly, it just needs to learn one more step forward.
        x = original_x + attn_output

        x = self.ln2(x)

        ffn_output = self.ffn(x)

        x = x + ffn_output # residual connection again

        return x


In [ ]:
# creating vision transformer

class ViT(nn.Module):
    def __init__(self, img_size, patch_size, num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout):
        # blk represents the number of transformer blocks
        super().__init__()
        # converting the inmage into patches
        self.patch_embedding = PatchEmbeddings(img_size, patch_size, num_hiddens)

        # develops a learnable classification token: a token at the end of the image tokens that provides a summary of what the image shows. This works because this token can see everything in the image and take the information to interpret the whole image. It also learns during the training process (hence the nn.Parameter)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, num_hiddens))

        num_patches = (img_size // patch_size) ** 2

        # learnable postion embedding (work to understand the positinal encoding of the image sections simlar to the M-RoPE)
        self.pos_embedding = nn.Parameter(torch.randn(1, num_patches + 1, num_hiddens))

        self.dropout = nn.Dropout(emb_dropout)

        self.blocks == nn.ModuleList(Block(num_hiddens, num_heads, blk_dropout, is_decoder=False) for _ in range(num_blks))

        self.layer_norm = nn.LayerNorm(num_hiddens)

    def forward(self, x):

        # creating the patches from the input image
        x = self.patch_embedding

        # change classification token shape to match the batch size
        cls_tokens = self.cls_toekn.expand(x.shape[0], -1, -1)

        # concatenate the classification token with the rest of the input patches
        x = torch.cat((cls_tokens, x), dim = 1)

        # add the position embedding (don't need to expand because the addition in pytorch accounts for mismatch in dimensions bbut torch.cat does not)
        x += self.pos_embedding

        x = self.dropout(x)

        # pass the embeddings through the transformer blocks
        for block in self.blocks:
            x = block(x)

        x = self.layer_norm(x[:, 0])

        return x

        


In [ ]:
# building the project that connects the vision model and the LLM

class MultiModalProjector(nn.Module):
    def __init__(self, n_embd, image_embd_dim, dropout=0.1):
        super().__init__()

        self.next = nn.Sequential(
            nn.Linear(image_embd_dim, 4*image_embd_dim), 

            nn.GELU(), # Gaussian Error Linear Unit -- used a lot in transformer development and it is because it helps smoothly scale input and it is continoutsly differentiable therefore the models can capute complex features and avoid the vanishing gradient problem

            nn.Linear(4 * image_embd_dim, n_embd), 

            nn.Dropout(dropout)
        )

    def forward(self, x):
        # pass the input through the projection network
        x = self.net(x)
        return x

In [ ]:
# decoder language model
class DecoderLanguageModel(nn.Module):
    def __init__(self, n_embd, image_embd_dim, vocab_size, num_heads, n_layer, use_images=False):
        super().__init__()

        self.use_images = use_images

        # token embedding table -- a lookup table that connects embedding values with values given to the tokens from the tokenizer
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        #position embedding table
        self.position_embedding_table = nn.Embedding(1000, n_embd)

        if use_images:
            self.image_projection = MultiModalProjector(n_embd, image_embd_dim)

        self.blocks = nn.Sequential(*[Block(n_embd, num_heads, is_decoder=True) for _ in range(n_layer)])

        self.ln_f = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, image_embds=None, targets=None):
        # get token embeddings from the input
        token_emb = self.token_embedding_table(idx)

        if self.use_images and image_embds is not None:
            # project and add image embeddings with token embeddings
            img_emb = self.image_project(image_embds).unsquess(1)
            tok_emb = torch.cat([img_emb, tok_emb], dim=1)

        pos_emb = self.position_embeding_table(torch.arrange(tok_emb.size(1), device=device)).unsqueeze(0)

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        # converts the output of the transformer into the vocabulary space to form actual words
        logits = self.lm_head(x)

        # targets holds the result that the large language model will need to output. However, it only holds answers for the text output. If an image was uploaded, its information was conated to the front of the text embeddings. This means that in order to calculte loss, targets is now the wrong size. This code works to fix it. Targets should not have any answer for the image data since that is only supposed to give context, hence it puts a bunch of -100 -- a number that is skipped when computing the cross entropy loss automatically.
        if targets is not None:
            if self.use_images and image_embds is not None:
                batch_size = idx.size(0)
                targets = torch.cat([torch.full((batch_size, 1), -100, dtype=torch.long, device=device), targets], dim=1)

            loss = F.cross_entropy(logits.view(-1, logits.zie(-1)), targets.view(-1), ignore_index=-100)

            return logits, loss
        return logits

    def generate(self, idx, image_embds, max_new_tokens):
        # autoregressive text generation
        B, T = idx.shape
        generated = idx

        if self.use_images and image_embds is not None:
            img_emb = self.image_projection(image_embds).unsqueeze(1)
            current_output = torch.cat([img_emb, self.token_embedding_table(idx)], dim=1)
        else:
            current_output = self.token_embedding_table(idx)

        for i in range(max_new_tokens):
            T_current = current_output.size(1)
            current_pos_emb = self.position_embedding_table(torch.arange(T_current, device =idx.device)).unsqueeze(0)
            current_output += current_pos_emb

            for block in self.blocks:
                current_output = block(current_output)

            logits = self.lm_head(current_output[:, -1, :])
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            generated = torch.cat((generated, idx_next), dim=1)
            idx_next_emb = self.token_embedding_table(idx_next)
            current_output = torch.cat((current_output, idx_next_emb), dim=1)

        return generated



In [ ]:
class VisionLanguageModel(nn.Module):
    def __init__(self, n_embd, image_embed_dim, vocab_size, n_layer, img_size, patch_size, num_heads, num_blks, emb_dropout, blk_dropout):
        super().__init__()
        
        # Set num_hiddens equal to image_embed_dim
        num_hiddens = image_embed_dim
        
        # Assert that num_hiddens is divisible by num_heads
        assert num_hiddens % num_heads == 0, "num_hiddens must be divisible by num_heads"
        
        # Initialize the vision encoder (ViT)
        self.vision_encoder = ViT(img_size, patch_size, num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout)
        
        # Initialize the language model decoder (DecoderLanguageModel)
        self.decoder = DecoderLanguageModel(n_embd, image_embed_dim, vocab_size, num_heads, n_layer, use_images=True)

    def forward(self, img_array, idx, targets=None):
        # Get the image embeddings from the vision encoder
        image_embeds = self.vision_encoder(img_array)
        
        # Check if the image embeddings are valid
        if image_embeds.nelement() == 0 or image_embeds.shape[1] == 0:
            raise ValueError("Something is wrong with the ViT model. It's returning an empty tensor or the embedding dimension is empty.")
        
        if targets is not None:
            # If targets are provided, compute the logits and loss
            logits, loss = self.decoder(idx, image_embeds, targets)
            return logits, loss
        else:
            # If targets are not provided, compute only the logits
            logits = self.decoder(idx, image_embeds)
            return logits

    def generate(self, img_array, idx, max_new_tokens):
        # Get the image embeddings from the vision encoder
        image_embeds = self.vision_encoder(img_array)
        
        # Check if the image embeddings are valid
        if image_embeds.nelement() == 0 or image_embeds.shape[1] == 0:
            raise ValueError("Something is wrong with the ViT model. It's returning an empty tensor or the embedding dimension is empty.")
        
        # Generate new tokens using the language model decoder
        generated_tokens = self.decoder.generate(idx, image_embeds, max_new_tokens)
        return generated_tokens
